# 01 - Adquisición de Datos y Exploración de Telemetría (Temporada 2025)
**Tesis de Maestría en Ciencia de Datos - Universidad Austral**  
*Autor: Nicolás Palavecino | Tutor: Ezequiel Nuske*

---

## 1. Introducción y Objetivos del Notebook
Este notebook constituye el primer paso del pipeline analítico:
1. Conexión con la API de telemetría de Fórmula 1 (**FastF1**) y activación del sistema de cache.
2. Carga de sesiones oficiales de la **Temporada 2025** (ej. *Italian Grand Prix - Monza*).
3. Filtrado de vueltas válidas y representativas (eliminación de vueltas bajo *Safety Car*, banderas amarillas, vueltas de entrada/salida a boxes).
4. Exploración de señales cinemáticas de alta frecuencia: velocidad (/h$), acelerador ($\%$), freno binario y aceleración longitudinal derivada (/s^2$).
5. Trazado del trazado del circuito con localización de vértices de curva (*apexes*).


In [ ]:
import fastf1 as ff1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from f1_style_clustering.config import load_config, load_circuit_config
from f1_style_clustering.data.loader import enable_cache, load_session, prepare_lap_telemetry
from f1_style_clustering.data.filters import filter_clean_laps
from f1_style_clustering.visualization.corner_maps import plot_circuit_with_zones

# Cargar configuraciones globales
config = load_config()
enable_cache(config['general']['cache_dir'])
print('FastF1 cache habilitado en:', config['general']['cache_dir'])


## 2. Carga de Sesión Oficial (Temporada 2025)

In [ ]:
# Cargar Gran Premio de Italia 2025 (Carrera)
SEASON = 2025
CIRCUIT = 'Italian Grand Prix'
SESSION_TYPE = 'R'

session = load_session(year=SEASON, grand_prix=CIRCUIT, session_type=SESSION_TYPE)
print(f'Evento: {session.event.EventName} | Sesión: {session.name} | Temporada: {session.event.year}')
print(f'Total de pilotos participantes: {len(session.drivers)}')


## 3. Filtrado y Validación de Vueltas Representativas
Para que el análisis del estilo de pilotaje sea riguroso, excluimos:
- Vueltas de entrada (*InLaps*) y salida (*OutLaps*) de boxes.
- Vueltas afectadas por banderas amarillas, rojas o regímenes de *Safety Car* / *Virtual Safety Car*.
- Vueltas no representativas (fuera de la ventana de *quicklaps*).


In [ ]:
# Resumen de vueltas limpias por piloto
drivers_summary = []
for drv in session.drivers:
    drv_info = session.get_driver(drv)
    abbr = drv_info.get('Abbreviation', str(drv))
    team = drv_info.get('TeamName', 'Unknown')
    
    all_laps = session.laps.pick_drivers(abbr)
    clean_laps = filter_clean_laps(session, driver=abbr, quick_laps_only=True, accurate_only=True)
    
    drivers_summary.append({
        'Driver': abbr,
        'Team': team,
        'Total_Laps': len(all_laps),
        'Clean_Representative_Laps': len(clean_laps),
    })

df_summary = pd.DataFrame(drivers_summary)
display(df_summary.head(10))


## 4. Trazado del Circuito y Señales Cinemáticas de Telemetría

In [ ]:
# Visualización del circuito con vértices
fig_circuit = plot_circuit_with_zones(session)
plt.show()


In [ ]:
# Comparación de telemetría de vuelta rápida
fastest_laps = session.laps.pick_quicklaps().sort_values(by='LapTime').groupby('Driver').first()
top_drivers = list(fastest_laps.index[:3])
print('Top 3 pilotos con vueltas más rápidas:', top_drivers)

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)

for drv in top_drivers:
    lap = session.laps.pick_drivers(drv).pick_fastest()
    tel = prepare_lap_telemetry(lap)
    if tel is not None:
        color = ff1.plotting.get_driver_color(drv, session=session) or '#00d2be'
        axes[0].plot(tel['Distance'], tel['Speed'], label=drv, color=color, lw=1.8)
        axes[1].plot(tel['Distance'], tel['Throttle'], color=color, lw=1.5)
        axes[2].plot(tel['Distance'], tel['Brake'], color=color, lw=1.5)

axes[0].set_ylabel('Velocidad (km/h)')
axes[0].set_title(f'Telemetría Comparativa de Vuelta Rápida - {session.event.EventName} {SEASON}', fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].grid(True, linestyle=':', alpha=0.5)

axes[1].set_ylabel('Acelerador (%)')
axes[1].grid(True, linestyle=':', alpha=0.5)

axes[2].set_ylabel('Freno')
axes[2].set_xlabel('Distancia en Pista (metros)')
axes[2].grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()
